# arb to excel

In [2]:
import json
import pathlib
import pandas as pd

# Carpeta donde están tus .arb (aquí uso el directorio actual)
ARB_DIR = pathlib.Path(".")

# Lista de archivos que queremos procesar
arb_files = [
    "app_el.arb",
    "app_en.arb",
    "app_es.arb",
    "app_fr.arb",
    "app_it.arb",
    "app_pt.arb",
    "app_tr.arb",
]

# Diccionario: lang_code -> { id -> texto }
translations = {}

for filename in arb_files:
    path = ARB_DIR / filename
    if not path.exists():
        print(f"Warning: {path} not found, skipping")
        continue
    
    # Idioma a partir del nombre de archivo: app_en.arb -> en
    lang = filename.split("_")[1].split(".")[0]  # "en", "es", ...
    
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # Nos quedamos solo con las claves "normales", sin las que empiezan por "@"
    lang_dict = {k: v for k, v in data.items() if not k.startswith("@")}
    translations[lang] = lang_dict

# Conjunto de todas las keys de todos los idiomas
all_keys = sorted({k for lang_dict in translations.values() for k in lang_dict.keys()})

# Construimos un DataFrame: primera columna "id", luego una columna por idioma
cols = ["id"] + sorted(translations.keys())  # por ejemplo: ["id", "el", "en", ...]
rows = []

for key in all_keys:
    row = {"id": key}
    for lang in translations.keys():
        row[lang] = translations[lang].get(key, "")
    rows.append(row)

df = pd.DataFrame(rows, columns=cols)

# Guardar a Excel
output_path = ARB_DIR / "translations.xlsx"
df.to_excel(output_path, index=False)

print(f"Excel generado en: {output_path}")


Excel generado en: translations.xlsx


# excel to arb

In [ ]:
import json
import pathlib
import pandas as pd
import math

# Carpeta donde están tus .arb y el Excel
ARB_DIR = pathlib.Path(".")

# Ruta del Excel generado antes
EXCEL_PATH = ARB_DIR / "translations.xlsx"

# Leer el Excel
df = pd.read_excel(EXCEL_PATH)

# Comprobamos que existe la columna 'id'
if "id" not in df.columns:
    raise ValueError("The Excel file must contain an 'id' column.")

# Todas las columnas de idioma (todas menos 'id')
lang_columns = [c for c in df.columns if c != "id"]

# Para cada idioma, actualizamos su ARB
for lang in lang_columns:
    arb_path = ARB_DIR / f"app_{lang}.arb"
    if not arb_path.exists():
        print(f"Warning: {arb_path} not found. A new file will be created.")
        data = {"@@locale": lang}
    else:
        with open(arb_path, "r", encoding="utf-8") as f:
            data = json.load(f)

    # Mapa id -> texto para este idioma
    lang_map = dict(zip(df["id"], df[lang]))

    # Actualizar solo las claves "normales" (no empiezan por '@')
    for key, value in lang_map.items():
        # Saltar NaN
        if isinstance(value, float) and math.isnan(value):
            continue

        # Si no hay texto, puedes decidir si dejarlo vacío o no tocarlo
        # Aquí lo dejamos vacío explícitamente:
        text = "" if value is None else str(value)

        # No tocamos las claves que empiezan por '@'
        if key.startswith("@"):
            continue

        data[key] = text

    # Guardar el ARB actualizado (preserva las claves @ tal cual)
    with open(arb_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Updated: {arb_path}")
